Data Cleaning in Pandas

In [4]:
import pandas as pd

In [8]:
#understand the dataset
# play_id : It's an additional number after an underscore '_' after the 'game_id' column from other datasets i.e. clean_game.csv
# e.g. game_id = 2015020201, play_id = '2015020201_227'
# Consideration: We can extract the game_id from the play_id
# strength : whether the number of players for the two teams are equal (even) or 1 team is short of typically, 1-2 players (power).
# game_winning_goal : whether the goal is a winning goal
# empty_net :  Team may take out its goalie during an event in a game 

# Error Msg : 'gameWinningGoal' column. Pandas is unable to deduce a single type for the entire column. Based on info() output, 
#              the column's dtype is object, which means likely contains a mix of strings, booleans, and possibly NaN values.

df_game_goals = pd.read_csv(r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\extract\raw_data\game_goals.csv")
df_game_goals.head()
df_game_goals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148992 entries, 0 to 148991
Data columns (total 4 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   play_id          148992 non-null  object
 1   strength         148992 non-null  object
 2   gameWinningGoal  147148 non-null  object
 3   emptyNet         143949 non-null  object
dtypes: object(4)
memory usage: 4.5+ MB


C:\Users\zacle\AppData\Local\Temp\ipykernel_9856\3992955804.py:12: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_game_goals = pd.read_csv(r"C:\Users\zacle\Desktop\Serene\Project\ETL\extract\raw_data\game_goals.csv")


In [10]:
# Create a new dataframe from df_game_goals (copying original to preserve data)
df_clean_game_goals = df_game_goals.copy()

Rename columns 

In [13]:
#Rename columns for consistency

import re

# Function to add an underscore before uppercase letters and convert to lowercase
def rename_columns(col_name):
    return re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', col_name).lower()

# Apply the function to all column names
df_clean_game_goals.columns = [rename_columns(col) for col in df_clean_game_goals.columns]
df_clean_game_goals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148992 entries, 0 to 148991
Data columns (total 4 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   play_id            148992 non-null  object
 1   strength           148992 non-null  object
 2   game_winning_goal  147148 non-null  object
 3   empty_net          143949 non-null  object
dtypes: object(4)
memory usage: 4.5+ MB


Check : Null 

In [16]:
# Null - All - check for missing or null values entire dataset
# Results : game_winning_goal = 1844
#           empty_net = 5043

# 'game_winning_goal' column with null values mean that 1. it is a goal 2. column will be populated with True regardless who scored
# 2. it's unclear whether the goal contributed to the final game-winning outcome, often because the game result was still in progress 
# at the time of that goal, or the goal didn’t end up being decisive.
# Action : it is possible to derive the NA values from game_plays.csv 

# 'empty_net' column with null values. In even play (when both teams have the same number of skaters on the ice, such as 5-on-5), there 
# should not be an empty net situation in a typical game scenario. We can assume that if the strength is 'even', to populate as "False"

df_clean_game_goals.isnull().sum()

play_id                 0
strength                0
game_winning_goal    1844
empty_net            5043
dtype: int64

In [18]:
#explore columns with null values
df_clean_game_goals[df_clean_game_goals.isnull().any(axis=1)]

,play_id,strength,game_winning_goal,empty_net
48,2015020346_400,Even,False,NaN
49,2015020346_402,Even,False,NaN
89,2016021020_327,Even,False,NaN
157,2015020086_314,Even,False,NaN
247,2017020930_349,Even,False,NaN
...,...,...,...,...
147933,2018021258_293,Even,False,NaN
147934,2018021258_295,Even,False,NaN
147945,2018021259_402,Even,False,NaN
147946,2018021259_403,Even,False,NaN


In [20]:
# 'empty_net' column with null values. In even play (when both teams have the same number of skaters on the ice, such as 5-on-5), there 
# should not be an empty net situation in a typical game scenario. We can assume that if the strength is 'even', to populate as "False"
# Results : 'empty_net' is resolved

df_clean_game_goals.loc[
    (df_clean_game_goals['strength'] == 'Even') & 
    (df_clean_game_goals['empty_net'].isnull()), 
    'empty_net'
] = False

df_clean_game_goals.isnull().sum()

play_id                 0
strength                0
game_winning_goal    1844
empty_net               0
dtype: int64

In [22]:
# 'game_winning_goals' column with null values. 
# Action : To delete the rows with null values. Since the deletion of 1844 records will not affect the data significantly as there are 148992 entries.
df_clean_game_goals = df_clean_game_goals.dropna(subset=['game_winning_goal'])
df_clean_game_goals

,play_id,strength,game_winning_goal,empty_net
0,2016020045_6,Even,False,False
1,2016020045_97,Even,False,False
2,2016020045_103,Power Play,False,False
3,2016020045_140,Power Play,False,False
4,2016020045_197,Power Play,False,False
...,...,...,...,...
148987,2018030417_81,Even,False,False
148988,2018030417_100,Even,True,False
148989,2018030417_257,Even,False,False
148990,2018030417_277,Even,False,False


Populate the null values in the game_winning_goal column using values from game_plays.csv Unsuccessful.

In [25]:
# 'game_winning_goal' rules 
# After the final score has been determined, the goal which leaves the winning Club one goal ahead of its opponent is the game-winning goal 
# (example: if Team A beats Team B 8-3, the player scoring the fourth goal for Team A receives credit for the game-winning goal).
# source : https://hockeyanswered.com/how-does-a-goalie-get-a-win-in-hockey/

# with null values with the assumption that 1. it is a goal regardless of the winning team 2. column 'game_winning_goal' 
# will be populated with True regardless who scored

# Identify the play that results in the winning goal based on the 'goals_away' and 'goals_home' columns in game_plays.csv 
# Update the 'game_winning_goal' column in df_clean_game_goals where it is currently null, based on the play_id from df_game_plays.
# We can add an additional column, which team wins
# actual game-winning goal based on when a team takes the lead and maintains it to the end of the game.

In [27]:
# Error - DtypeWarning: 'game_winning_goal' column have mixed types, typically occurs when a column contains values of different data types 
# (e.g., both numbers and strings) or has missing values. 
# Check the column for data types
# Result : After resolving the null values (String dtype). 
# 'game_winning_goal' column have True and False, Boolean dtype results only.

# Check unique values in the 'game_winning_goal' column
df_clean_game_goals['game_winning_goal'].unique()

array([False, True], dtype=object)

Check : Duplicates

In [30]:
# Count unique 'play_id' values
unique_ids = df_clean_game_goals['play_id'].nunique()

# Count total number of rows
total_rows = len(df_clean_game_goals)

# Display the results
print(f"Unique game_ids: {unique_ids}")
print(f"Total rows: {total_rows}")
print(f"There are {total_rows - unique_ids} rows with duplicates.")

# Results: Unique game_ids: 131501 / Total rows: 147148 / There are 15647 rows with duplicates.

Unique game_ids: 131501
Total rows: 147148
There are 15647 rows with duplicates.


In [32]:
# Result : 15647 duplicates
# Additional Duplicate Check for 'play_id', 'strength' combinations
duplicates_play_strength = df_clean_game_goals[df_clean_game_goals.duplicated(subset=['play_id', 'strength','game_winning_goal'], keep=False)]

# Sort rows with duplicated results in ascending order by 'play_id' and 'strength'
duplicates_play_strength_sorted = duplicates_play_strength.sort_values(by=['play_id', 'strength'], ascending=True)

# Count total number of rows with duplicates based on 'play_id' and 'strength'
total_duplicates_play_strength = df_clean_game_goals.duplicated(subset=['play_id', 'strength'], keep=False).sum()

# Count the unique combinations of 'play_id' and 'strength'
unique_play_strength_combinations = df_clean_game_goals[['play_id', 'strength']].drop_duplicates().shape[0]

# Count total number of rows
total_rows = len(df_clean_game_goals)

# Display the results
print(f"Unique 'play_id' and 'strength' combinations: {unique_play_strength_combinations}")
print(f"Total rows: {total_rows}")
print(f"There are {total_rows - unique_play_strength_combinations} rows with duplicates based on 'play_id' and 'strength'.")

# Display the sorted duplicates for further review
print("Sorted Duplicates based on 'play_id' and 'strength':")
print(duplicates_play_strength_sorted[['play_id', 'strength']])


Unique 'play_id' and 'strength' combinations: 131501
Total rows: 147148
There are 15647 rows with duplicates based on 'play_id' and 'strength'.
Sorted Duplicates based on 'play_id' and 'strength':
               play_id    strength
132442  2018020001_213        Even
132468  2018020001_213        Even
132443  2018020001_219  Power Play
132469  2018020001_219  Power Play
132444  2018020001_363        Even
...                ...         ...
127280   2019040653_68        Even
127243   2019040653_82        Even
127281   2019040653_82        Even
127236    2019040653_9        Even
127274    2019040653_9        Even

[31294 rows x 2 columns]


In [34]:
# Remove duplicates based on 'play_id' and 'strength'
df_clean_game_goals = df_clean_game_goals.drop_duplicates(subset=['play_id', 'strength'], keep='first')

# Display the cleaned dataframe (first 5 rows as an example)
df_clean_game_goals

,play_id,strength,game_winning_goal,empty_net
0,2016020045_6,Even,False,False
1,2016020045_97,Even,False,False
2,2016020045_103,Power Play,False,False
3,2016020045_140,Power Play,False,False
4,2016020045_197,Power Play,False,False
...,...,...,...,...
148982,2018030417_81,Even,False,False
148983,2018030417_100,Even,True,False
148984,2018030417_257,Even,False,False
148985,2018030417_277,Even,False,False


In [40]:
# Final Duplicate Check: Check for duplicates based on 'play_id' and 'strength'
duplicates_play_id = df_clean_game_goals[df_clean_game_goals.duplicated(subset=['play_id', 'strength'], keep=False)]

if not duplicates_play_id.empty:
    # Sort the duplicated rows by 'play_id' and 'strength'
    duplicates_play_id_sorted = duplicates_play_id.sort_values(by=['play_id', 'strength'], ascending=True)

    # Print the sorted duplicated rows
    print("Sorted Duplicates based on 'play_id' and 'strength':")
    print(duplicates_play_id_sorted[['play_id', 'strength']])
else:
    print("No duplicates found based on 'play_id' and 'strength'. Data is clean!")

No duplicates found based on 'play_id' and 'strength'. Data is clean!


Change Data Type 

In [43]:
# Change data type syntax - df['column_name'] = df['column_name'].astype('desired_data_type')
# Redefine the game_winning_goal column. There was an error earlier on due to mixed Boolean and String datatype in 'game_winning_goal'. 
# Though it was resolved. Keep all values as string to avoid complications later on during the analysis process
# Change data type syntax - df['column_name'] = df['column_name'].astype('desired_data_type')
# Use .loc to explicitly update the columns
df_clean_game_goals.loc[:, 'game_winning_goal'] = df_clean_game_goals['game_winning_goal'].astype(str)
df_clean_game_goals.loc[:, 'empty_net'] = df_clean_game_goals['empty_net'].astype(str)

In [45]:
df_clean_game_goals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 131501 entries, 0 to 148986
Data columns (total 4 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   play_id            131501 non-null  object
 1   strength           131501 non-null  object
 2   game_winning_goal  131501 non-null  object
 3   empty_net          131501 non-null  object
dtypes: object(4)
memory usage: 5.0+ MB


In [47]:
#save file locally
df_clean_game_goals.to_csv(r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_goals.csv", index=False)

In [51]:
#Issue : game_winning_goal and empty_net convert back to boolean 
# During the loading stage to PostgreSQL, must parse the string values.
df = pd.read_csv(r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_goals.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131501 entries, 0 to 131500
Data columns (total 4 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   play_id            131501 non-null  object
 1   strength           131501 non-null  object
 2   game_winning_goal  131501 non-null  bool  
 3   empty_net          131501 non-null  bool  
dtypes: bool(2), object(2)
memory usage: 2.3+ MB
